[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/C.%20Transportation%2C%20Routing%20%26%20Freight%20Management%20%28The%20Physical%20Internet%29/072.%20The%20Capacitated%20VRP%20%28CVRP%29/P72-Tier-6_executed.ipynb)

---


# 72. The Capacitated Vehicle Routing Problem (CVRP)

## Tier 6: An Autonomous Multi-Agent System

### Purpose and workflow

This notebook illustrates how autonomous agents can coordinate delivery-task assignments using the Contract Net Protocol. A coordinator announces each task, vehicle agents submit feasible cost-based bids, and the lowest-price bid is awarded. The awarded tasks are then grouped by vehicle and summarized. Communication and coordination indicators are calculated from this simulated process.

The central teaching goal is to connect agent roles, messages, bidding, task allocation, and system-level metrics in one executable example. This is a coordination simulation, not an exact CVRP optimizer: the bidding cost is a simplified proxy and does not evaluate complete route distances.

### Background: CVRP and agent coordination

In the capacitated vehicle routing problem, a fleet serves customers with known demands. Each vehicle starts and ends at a depot, each customer is served, and the total demand assigned to a vehicle cannot exceed its capacity. A classical CVRP objective often minimizes total travel distance. This notebook focuses instead on the distributed task-allocation process: vehicles act as agents, delivery requests become tasks, and bids express estimated task costs.

The Contract Net Protocol is a task-allocation procedure. A manager announces a task; eligible participants respond with proposals; the manager selects a proposal and awards the task. Here, the coordination agent acts as manager, vehicle agents are bidders, and the lowest feasible bid wins. A market subsystem records tasks, bids, and awards, but it does not run a separate auction to select winners.

For task $j$ and vehicle $i$, let $d_j$ be the task demand, $C_i$ the vehicle capacity, and $L_i$ the load already assigned to that vehicle. The capacity-feasibility rule is

$$
L_i + d_j \leq C_i.
$$

Among vehicles that satisfy this rule, the coordinator awards the task to the vehicle with the smallest submitted bid $b_{ij}$. Tasks are processed sequentially, so each award updates the vehicle's assigned load before the next task is considered. This prevents the sum of awarded demands from exceeding a vehicle's capacity.

The bid is a deliberately simple estimate:

$$
\begin{aligned}
c_{ij} &= c^{\text{distance}}_{ij} + c^{\text{time}}_{ij} + c^{\text{resource}}_{ij}, \\
b_{ij} &= c_{ij}(1 + m),
\end{aligned}
$$

where $c_{ij}$ is the estimated cost and $m$ is the fixed profit margin. In the code, the distance and time components are constant proxies; the resource component depends on demand. Therefore, bids do not represent route-dependent marginal costs, and the resulting assignments should not be interpreted as a minimum-distance CVRP solution.

### Model assumptions and scope

- Vehicle capacities and customer demands are expressed in the same abstract load units.
- Tasks are considered in the order supplied. Each vehicle's remaining capacity is updated immediately after an award.
- Each task is assigned independently using the current remaining capacity and the lowest feasible bid. The simulation does not optimize the order in which a vehicle visits its customers.
- Customer coordinates, deadlines, and service priorities are stored in task data but are not used to estimate travel, enforce time windows, or rank bids.
- The simulation treats awarded contracts as completed during its execution step. It does not model road networks, vehicle movement, uncertain travel times, failures, or live communication delays.
- Adaptation metrics are calculated from agent histories, but this example does not train agents or add performance observations. Consequently, those metrics describe the available history rather than demonstrated learning.
- Communication overhead is a constructed proxy based on message content and, when both agents have locations, their Euclidean separation. It is not a measured network cost or a reduction relative to a baseline.

### What to examine

Follow a task from creation through announcement, bid generation, award, and simulated completion. Then inspect the report and plots. Coordination indicators are descriptive measures defined by this simulation; they are not universal measures of multi-agent-system quality. In particular, no empirical percentages or performance improvements are assumed in advance.

In [ ]:
from dataclasses import dataclass
from enum import Enum
from typing import Any, Dict, List, Optional, Tuple
import random
import time

import matplotlib.pyplot as plt
import numpy as np

# Fixed seeds make the generated agent and customer data reproducible.
np.random.seed(42)
random.seed(42)

print("Libraries imported successfully for the multi-agent CVRP simulation.")

In [ ]:
class AgentType(Enum):
    VEHICLE_AGENT = "vehicle_agent"
    CUSTOMER_AGENT = "customer_agent"
    COORDINATION_AGENT = "coordination_agent"
    MARKET_AGENT = "market_agent"
    MONITORING_AGENT = "monitoring_agent"


class AgentState(Enum):
    IDLE = "idle"
    ACTIVE = "active"
    NEGOTIATING = "negotiating"
    EXECUTING = "executing"
    COMMUNICATING = "communicating"
    LEARNING = "learning"


class MessageType(Enum):
    TASK_ANNOUNCEMENT = "task_announcement"
    BID_PROPOSAL = "bid_proposal"
    AWARD_NOTIFICATION = "award_notification"
    TASK_COMPLETION = "task_completion"
    STATUS_UPDATE = "status_update"
    NEGOTIATION_REQUEST = "negotiation_request"
    MARKET_BID = "market_bid"
    COORDINATION_MESSAGE = "coordination_message"


@dataclass
class Message:
    """A message exchanged between agents."""

    message_id: str
    sender_id: str
    receiver_id: str
    message_type: MessageType
    content: Dict[str, Any]
    timestamp: float
    priority: int

    def __post_init__(self):
        self.delivery_status = "pending"
        self.retry_count = 0
        self.max_retries = 3


@dataclass
class Agent:
    """Base data structure shared by the agent roles."""

    agent_id: str
    agent_type: AgentType
    state: AgentState
    capabilities: List[str]
    resources: Dict[str, float]

    def __post_init__(self):
        self.message_queue = []
        self.neighbors = []
        self.performance_history = []
        self.learning_rate = 0.1
        self.communication_overhead = 0.0


@dataclass
class VehicleAgent(Agent):
    """Vehicle agent with a capacity and an assigned route."""

    vehicle_id: int
    capacity: int
    current_location: Tuple[float, float]
    current_route: List[int]
    current_load: int
    fuel_level: float

    def __post_init__(self):
        super().__post_init__()
        self.total_distance_traveled = 0.0
        self.tasks_completed = 0
        self.bid_strategy = "cost_minimization"
        self.negotiation_power = 0.5


@dataclass
class CustomerAgent(Agent):
    """Customer agent representing a delivery request."""

    customer_id: int
    location: Tuple[float, float]
    demand: int
    service_priority: int
    time_window: Tuple[float, float]

    def __post_init__(self):
        super().__post_init__()
        self.service_history = []
        self.satisfaction_score = 0.5
        self.waiting_time = 0.0


@dataclass
class Task:
    """A delivery task offered to vehicle agents."""

    task_id: str
    task_type: str
    requirements: Dict[str, Any]
    reward: float
    deadline: float

    def __post_init__(self):
        self.assigned_agent = None
        self.bids_received = []
        self.status = "open"
        self.creation_time = time.time()


@dataclass
class MultiAgentSystem:
    """Shared state for the agents and their coordination process."""

    system_id: str
    agents: List[Agent]
    communication_protocol: str
    coordination_mechanism: str

    def __post_init__(self):
        self.message_bus = []
        self.task_market = []
        self.performance_metrics = {
            "coordination_efficiency": 0.0,
            "communication_overhead": 0.0,
            "emergent_optimization": 0.0,
            "system_throughput": 0.0,
        }
        self.system_history = []


print("Multi-agent system data structures defined.")

In [ ]:
class AgentCommunicationSystem:
    """Routes messages and implements task-allocation protocols."""

    def __init__(self, multi_agent_system: MultiAgentSystem):
        self.mas = multi_agent_system
        self.communication_protocols = {
            "contract_net": self._contract_net_protocol,
            "market_based": self._market_based_protocol,
            "direct_negotiation": self._direct_negotiation_protocol,
        }
        self.message_routing_table = {}
        self.communication_stats = {
            "messages_sent": 0,
            "messages_delivered": 0,
            "average_latency": 0.0,
            "failed_deliveries": 0,
        }

    def send_message(self, message: Message) -> bool:
        """Place a message on the bus and attempt delivery."""
        self.mas.message_bus.append(message)
        self.communication_stats["messages_sent"] += 1
        success = self._route_message(message)
        if success:
            self.communication_stats["messages_delivered"] += 1
            message.delivery_status = "delivered"
        else:
            self.communication_stats["failed_deliveries"] += 1
            message.delivery_status = "failed"
        return success

    def _route_message(self, message: Message) -> bool:
        """Deliver a message to the named receiver, if it exists."""
        receiver_agent = next(
            (agent for agent in self.mas.agents
             if agent.agent_id == message.receiver_id),
            None,
        )
        if receiver_agent is None:
            return False

        receiver_agent.message_queue.append(message)
        sender_agent = next(
            (agent for agent in self.mas.agents
             if agent.agent_id == message.sender_id),
            None,
        )
        if sender_agent is not None:
            sender_agent.communication_overhead += (
                self._calculate_communication_overhead(message)
            )
        return True

    def _calculate_communication_overhead(self, message: Message) -> float:
        """Estimate communication overhead with a simple proxy."""
        base_overhead = 0.1
        sender_agent = next(
            (agent for agent in self.mas.agents
             if agent.agent_id == message.sender_id),
            None,
        )
        receiver_agent = next(
            (agent for agent in self.mas.agents
             if agent.agent_id == message.receiver_id),
            None,
        )

        if (sender_agent is not None and receiver_agent is not None
                and hasattr(sender_agent, "current_location")
                and hasattr(receiver_agent, "current_location")):
            distance = np.linalg.norm(
                np.asarray(sender_agent.current_location)
                - np.asarray(receiver_agent.current_location)
            )
            distance_overhead = distance * 0.01
        else:
            distance_overhead = 0.2

        complexity_overhead = len(str(message.content)) * 0.001
        return base_overhead + distance_overhead + complexity_overhead

    def _contract_net_protocol(self, task: Task) -> List[Message]:
        """Announce a task and collect feasible vehicle bids."""
        messages = [
            Message(
                message_id=f"announce_{task.task_id}",
                sender_id="coordination_agent",
                receiver_id="broadcast",
                message_type=MessageType.TASK_ANNOUNCEMENT,
                content={
                    "task_id": task.task_id,
                    "task_type": task.task_type,
                    "requirements": task.requirements,
                    "reward": task.reward,
                    "deadline": task.deadline,
                },
                timestamp=time.time(),
                priority=5,
            )
        ]
        vehicle_agents = [
            agent for agent in self.mas.agents
            if agent.agent_type == AgentType.VEHICLE_AGENT
        ]
        for agent in vehicle_agents:
            bid = self._generate_bid(agent, task)
            if bid is not None:
                messages.append(
                    Message(
                        message_id=f"bid_{task.task_id}_{agent.agent_id}",
                        sender_id=agent.agent_id,
                        receiver_id="coordination_agent",
                        message_type=MessageType.BID_PROPOSAL,
                        content=bid,
                        timestamp=time.time(),
                        priority=6,
                    )
                )
        return messages

    def _market_based_protocol(self, task: Task) -> List[Message]:
        """Create a market announcement for a task."""
        return [
            Message(
                message_id=f"market_{task.task_id}",
                sender_id="market_agent",
                receiver_id="broadcast",
                message_type=MessageType.MARKET_BID,
                content={
                    "market_type": "double_auction",
                    "task_id": task.task_id,
                    "starting_price": task.reward * 0.8,
                    "reserve_price": task.reward * 0.6,
                },
                timestamp=time.time(),
                priority=4,
            )
        ]

    def _direct_negotiation_protocol(self, task: Task) -> List[Message]:
        """Request proposals from the most experienced vehicle agents."""
        vehicle_agents = sorted(
            [agent for agent in self.mas.agents
             if agent.agent_type == AgentType.VEHICLE_AGENT],
            key=lambda agent: len(agent.performance_history),
            reverse=True,
        )
        messages = []
        for index, agent in enumerate(vehicle_agents[:3]):
            messages.append(
                Message(
                    message_id=f"negotiate_{task.task_id}_{agent.agent_id}",
                    sender_id="coordination_agent",
                    receiver_id=agent.agent_id,
                    message_type=MessageType.NEGOTIATION_REQUEST,
                    content={
                        "task_id": task.task_id,
                        "negotiation_power": agent.negotiation_power,
                        "proposed_reward": task.reward * (1 - index * 0.1),
                    },
                    timestamp=time.time(),
                    priority=7,
                )
            )
        return messages

    def _generate_bid(
        self, agent: VehicleAgent, task: Task
    ) -> Optional[Dict[str, Any]]:
        """Return a cost-based bid only when capacity and reward allow it."""
        demand = task.requirements.get("capacity", 0)
        if demand <= 0 or agent.current_load + demand > agent.capacity:
            return None
        if task.reward <= 0:
            return None

        distance_cost = self._estimate_distance_cost(agent, task)
        time_cost = self._estimate_time_cost(agent, task)
        resource_cost = self._estimate_resource_cost(agent, task)
        total_cost = distance_cost + time_cost + resource_cost
        bid_price = total_cost * 1.2
        if bid_price > task.reward:
            return None

        return {
            "agent_id": agent.agent_id,
            "bid_price": bid_price,
            "estimated_cost": total_cost,
            "completion_time": self._estimate_time_cost(agent, task),
            "confidence": 1.0 - total_cost / task.reward,
            "capabilities": agent.capabilities,
        }

    def _estimate_distance_cost(
        self, agent: VehicleAgent, task: Task
    ) -> float:
        """Use the stated constant distance-cost proxy."""
        return 10.0 * 0.5

    def _estimate_time_cost(
        self, agent: VehicleAgent, task: Task
    ) -> float:
        """Use the stated constant time-cost proxy, in cost units."""
        return 2.0 * 10.0

    def _estimate_resource_cost(
        self, agent: VehicleAgent, task: Task
    ) -> float:
        """Estimate resource cost in proportion to task demand."""
        return task.requirements.get("capacity", 1) * 2.0

    def get_communication_stats(self) -> Dict[str, float]:
        """Summarize delivery rates and the overhead proxy."""
        total_messages = self.communication_stats["messages_sent"]
        if total_messages:
            delivery_rate = (
                self.communication_stats["messages_delivered"] / total_messages
            )
            failure_rate = (
                self.communication_stats["failed_deliveries"] / total_messages
            )
        else:
            delivery_rate = 0.0
            failure_rate = 0.0

        agents = self.mas.agents
        total_overhead = sum(
            agent.communication_overhead
            for agent in agents
            if hasattr(agent, "communication_overhead")
        )
        average_overhead = total_overhead / len(agents) if agents else 0.0
        return {
            "total_messages": total_messages,
            "delivery_rate": delivery_rate,
            "failure_rate": failure_rate,
            "average_overhead": average_overhead,
        }


print("Agent communication and Contract Net Protocol components defined.")

In [ ]:
class MarketBasedCoordination:
    """Store market activity and provide simple allocation statistics."""

    def __init__(self, multi_agent_system: MultiAgentSystem):
        self.mas = multi_agent_system
        self.marketplace = {
            "tasks": [],
            "bids": [],
            "contracts": [],
            "prices": [],
        }
        self.market_mechanisms = {
            "double_auction": self._double_auction,
            "continuous_double_auction": self._continuous_double_auction,
            "combinatorial_auction": self._combinatorial_auction,
        }
        self.market_stats = {
            "total_transactions": 0,
            "average_price": 0.0,
            "market_efficiency": 0.0,
            "price_volatility": 0.0,
        }

    def submit_task_to_market(self, task: Task) -> None:
        """Register a task and notify market agents directly."""
        self.marketplace["tasks"].append(task)
        market_agents = [
            agent for agent in self.mas.agents
            if agent.agent_type == AgentType.MARKET_AGENT
        ]
        for agent in market_agents:
            agent.message_queue.append(
                Message(
                    message_id=f"market_notification_{task.task_id}",
                    sender_id="marketplace",
                    receiver_id=agent.agent_id,
                    message_type=MessageType.COORDINATION_MESSAGE,
                    content={"action": "new_task", "task_id": task.task_id},
                    timestamp=time.time(),
                    priority=4,
                )
            )

    def submit_bid_to_market(self, bid: Dict[str, Any]) -> None:
        """Record a bid so that its task and price can be inspected."""
        self.marketplace["bids"].append(bid)
        self.marketplace["prices"].append(bid["bid_price"])

    def _double_auction(self, task: Task) -> Optional[Dict[str, Any]]:
        """Select the lowest-price recorded bid for a task."""
        task_bids = [
            bid for bid in self.marketplace["bids"]
            if bid.get("task_id") == task.task_id
        ]
        if not task_bids:
            return None

        winning_bid = min(task_bids, key=lambda bid: bid["bid_price"])
        contract = {
            "task_id": task.task_id,
            "winner_id": winning_bid["agent_id"],
            "contract_price": winning_bid["bid_price"],
            "execution_time": winning_bid["completion_time"],
            "contract_status": "active",
            "creation_time": time.time(),
        }
        self.marketplace["contracts"].append(contract)
        self.market_stats["total_transactions"] += 1
        prices = [item["contract_price"] for item in self.marketplace["contracts"]]
        self.market_stats["average_price"] = float(np.mean(prices))
        return contract

    def _continuous_double_auction(
        self, task: Task
    ) -> Optional[Dict[str, Any]]:
        """Use the same lowest-bid rule for this illustrative interface."""
        return self._double_auction(task)

    def _combinatorial_auction(self, tasks: List[Task]) -> List[Dict[str, Any]]:
        """Apply the task-level bid selection independently to each task."""
        contracts = []
        for task in tasks:
            contract = self._double_auction(task)
            if contract is not None:
                contracts.append(contract)
        return contracts

    def optimize_market_allocation(self) -> Dict[str, Any]:
        """Report allocation coverage and descriptive price variation."""
        tasks = self.marketplace["tasks"]
        contracts = self.marketplace["contracts"]
        awarded_task_ids = {contract["task_id"] for contract in contracts}
        if tasks:
            efficiency = len(awarded_task_ids) / len(tasks)
        else:
            efficiency = 0.0

        prices = [contract["contract_price"] for contract in contracts]
        if len(prices) > 1 and np.mean(prices) != 0:
            price_volatility = float(np.std(prices) / np.mean(prices))
        else:
            price_volatility = 0.0
        average_price = float(np.mean(prices)) if prices else 0.0
        self.market_stats.update(
            {
                "total_transactions": len(contracts),
                "average_price": average_price,
                "market_efficiency": efficiency,
                "price_volatility": price_volatility,
            }
        )
        return {
            "market_efficiency": efficiency,
            "price_volatility": price_volatility,
            "total_transactions": len(contracts),
            "average_price": average_price,
            "active_contracts": sum(
                contract.get("contract_status") == "active"
                for contract in contracts
            ),
        }


print("Market activity and allocation statistics defined.")

In [ ]:
class EmergentBehaviorSystem:
    """Calculate descriptive indicators from recorded agent activity."""

    def __init__(self, multi_agent_system: MultiAgentSystem):
        self.mas = multi_agent_system
        self.behavior_patterns = []
        self.emergence_metrics = {
            "coordination_level": 0.0,
            "adaptation_rate": 0.0,
            "self_organization": 0.0,
            "collective_intelligence": 0.0,
        }

    def observe_emergent_behavior(self) -> Dict[str, Any]:
        """Update indicators using interactions, awards, and histories."""
        interaction_patterns = self._analyze_interactions()
        coordination_patterns = self._analyze_coordination()
        adaptation_patterns = self._analyze_adaptation()

        self.emergence_metrics["coordination_level"] = (
            coordination_patterns["coordination_score"]
        )
        self.emergence_metrics["adaptation_rate"] = (
            adaptation_patterns["adaptation_score"]
        )
        self.emergence_metrics["self_organization"] = (
            interaction_patterns["organization_score"]
        )
        self.emergence_metrics["collective_intelligence"] = (
            self._calculate_collective_intelligence()
        )
        return {
            "interaction_patterns": interaction_patterns,
            "coordination_patterns": coordination_patterns,
            "adaptation_patterns": adaptation_patterns,
            "emergence_metrics": self.emergence_metrics.copy(),
        }

    def _analyze_interactions(self) -> Dict[str, Any]:
        """Count messages and measure the share of coordination and bid messages."""
        total_messages = len(self.mas.message_bus)
        message_types = {}
        for message in self.mas.message_bus:
            name = message.message_type.value
            message_types[name] = message_types.get(name, 0) + 1

        if total_messages:
            organization_score = (
                message_types.get("coordination_message", 0)
                + message_types.get("bid_proposal", 0)
            ) / total_messages
        else:
            organization_score = 0.0
        return {
            "total_interactions": total_messages,
            "message_types": message_types,
            "organization_score": organization_score,
            "interaction_density": (
                total_messages / len(self.mas.agents) if self.mas.agents else 0.0
            ),
        }

    def _analyze_coordination(self) -> Dict[str, Any]:
        """Relate awarded tasks to the balance of vehicle workloads."""
        active_contracts = sum(
            contract.get("contract_status") == "active"
            for contract in self.mas.task_market
        )
        vehicle_agents = [
            agent for agent in self.mas.agents
            if agent.agent_type == AgentType.VEHICLE_AGENT
        ]
        workloads = [len(agent.current_route) for agent in vehicle_agents]
        if workloads and np.mean(workloads) > 0:
            workload_balance = max(
                0.0, 1.0 - float(np.std(workloads) / np.mean(workloads))
            )
        else:
            workload_balance = 0.0
        coordination_score = (
            active_contracts / max(1, len(vehicle_agents)) * workload_balance
        )
        return {
            "active_contracts": active_contracts,
            "workload_balance": workload_balance,
            "coordination_score": coordination_score,
            "agent_participation": (
                len(vehicle_agents) / len(self.mas.agents)
                if self.mas.agents else 0.0
            ),
        }

    def _analyze_adaptation(self) -> Dict[str, Any]:
        """Estimate change in recorded performance histories, if available."""
        learning_agents = 0
        total_learning = 0.0
        for agent in self.mas.agents:
            history = getattr(agent, "performance_history", [])
            if history:
                learning_agents += 1
                if len(history) > 1:
                    recent = float(np.mean(history[-5:]))
                    early = float(np.mean(history[:5]))
                    if early > 0:
                        total_learning += max(0.0, (recent - early) / early)

        adaptation_score = (
            total_learning / learning_agents if learning_agents else 0.0
        )
        return {
            "learning_agents": learning_agents,
            "total_learning": total_learning,
            "adaptation_score": adaptation_score,
            "learning_rate": (
                total_learning / len(self.mas.agents) if self.mas.agents else 0.0
            ),
        }

    def _calculate_collective_intelligence(self) -> float:
        """Combine the three descriptive indicators with stated weights."""
        metrics = self.emergence_metrics
        return (
            metrics["coordination_level"] * 0.4
            + metrics["adaptation_rate"] * 0.3
            + metrics["self_organization"] * 0.3
        )

    def identify_emergent_patterns(self) -> List[str]:
        """Return labels for indicators that exceed their chosen thresholds."""
        patterns = []
        if self.emergence_metrics["coordination_level"] > 0.8:
            patterns.append("High coordination indicator")
        if self.emergence_metrics["self_organization"] > 0.7:
            patterns.append("High message-organization indicator")
        if self.emergence_metrics["adaptation_rate"] > 0.6:
            patterns.append("High adaptation indicator")
        if self.emergence_metrics["collective_intelligence"] > 0.7:
            patterns.append("High combined indicator")
        return patterns


print("Emergent-behavior indicators defined.")

In [ ]:
class MultiAgentCVRPSystem:
    """Coordinate CVRP delivery tasks through Contract Net bidding."""

    def __init__(self):
        self.mas = MultiAgentSystem(
            system_id="cvrp_multi_agent_001",
            agents=[],
            communication_protocol="contract_net",
            coordination_mechanism="market_based",
        )
        self.communication_system = AgentCommunicationSystem(self.mas)
        self.market_coordination = MarketBasedCoordination(self.mas)
        self.emergent_behavior = EmergentBehaviorSystem(self.mas)
        self._initialize_agents()
        self.system_history = []

    def _initialize_agents(self):
        """Create the vehicle, customer, coordination, market, and monitoring agents."""
        for index in range(1, 9):
            self.mas.agents.append(
                VehicleAgent(
                    agent_id=f"vehicle_agent_{index}",
                    agent_type=AgentType.VEHICLE_AGENT,
                    state=AgentState.IDLE,
                    capabilities=["routing", "transportation", "negotiation"],
                    resources={"capacity": 100, "fuel": 100.0, "time": 8.0},
                    vehicle_id=index,
                    capacity=100,
                    current_location=(
                        float(np.random.uniform(0, 20)),
                        float(np.random.uniform(0, 20)),
                    ),
                    current_route=[],
                    current_load=0,
                    fuel_level=100.0,
                )
            )

        for index in range(1, 13):
            self.mas.agents.append(
                CustomerAgent(
                    agent_id=f"customer_agent_{index}",
                    agent_type=AgentType.CUSTOMER_AGENT,
                    state=AgentState.IDLE,
                    capabilities=["demand_generation", "service_evaluation"],
                    resources={
                        "budget": 1000.0,
                        "priority": float(np.random.uniform(0.3, 1.0)),
                    },
                    customer_id=index,
                    location=(
                        float(np.random.uniform(0, 20)),
                        float(np.random.uniform(0, 20)),
                    ),
                    demand=int(np.random.randint(5, 25)),
                    service_priority=int(np.random.randint(1, 4)),
                    time_window=(0.0, 8.0),
                )
            )

        agent_specs = [
            (
                "coordination_agent",
                AgentType.COORDINATION_AGENT,
                ["task_allocation", "contract_management", "system_optimization"],
                {"processing_power": 100.0, "memory": 1000.0},
            ),
            (
                "market_agent",
                AgentType.MARKET_AGENT,
                ["price_setting", "market_clearing", "resource_allocation"],
                {"market_data": 100.0, "analytical_power": 50.0},
            ),
            (
                "monitoring_agent",
                AgentType.MONITORING_AGENT,
                ["performance_monitoring", "behavior_analysis", "system_health"],
                {"monitoring_capacity": 100.0, "analysis_power": 75.0},
            ),
        ]
        for agent_id, agent_type, capabilities, resources in agent_specs:
            self.mas.agents.append(
                Agent(
                    agent_id=agent_id,
                    agent_type=agent_type,
                    state=AgentState.ACTIVE,
                    capabilities=capabilities,
                    resources=resources,
                )
            )

    def solve_cvrp_multi_agent(
        self, problem_instance: Dict[str, Any]
    ) -> Dict[str, Any]:
        """Create tasks, allocate them, simulate execution, and report indicators."""
        tasks = self._create_tasks_from_problem(problem_instance)
        print("=" * 70)
        print("MULTI-AGENT CVRP TASK ALLOCATION")
        print("=" * 70)
        print("\nProblem configuration:")
        print(f"  Total agents: {len(self.mas.agents)}")
        print(
            "  Vehicle agents: "
            f"{sum(a.agent_type == AgentType.VEHICLE_AGENT for a in self.mas.agents)}"
        )
        print(
            "  Customer agents: "
            f"{sum(a.agent_type == AgentType.CUSTOMER_AGENT for a in self.mas.agents)}"
        )
        print(f"  Delivery tasks: {len(tasks)}")

        for task in tasks:
            self.market_coordination.submit_task_to_market(task)
        contracts = self._run_task_allocation(tasks)
        execution_results = self._execute_contracts(contracts)
        emergent_analysis = self.emergent_behavior.observe_emergent_behavior()
        system_performance = self._calculate_system_performance(
            contracts, emergent_analysis
        )
        return {
            "contracts": contracts,
            "execution_results": execution_results,
            "emergent_analysis": emergent_analysis,
            "system_performance": system_performance,
        }

    def _create_tasks_from_problem(
        self, problem_instance: Dict[str, Any]
    ) -> List[Task]:
        """Represent each customer request as one delivery task."""
        tasks = []
        for customer in problem_instance.get("customers", []):
            demand = customer["demand"]
            if demand <= 0:
                raise ValueError("Customer demand must be positive.")
            tasks.append(
                Task(
                    task_id=f"task_{customer['id']}",
                    task_type="delivery",
                    requirements={
                        "customer_id": customer["id"],
                        "location": customer["location"],
                        "demand": demand,
                        "capacity": demand,
                        "priority": customer.get("priority", 1),
                    },
                    reward=demand * 10,
                    deadline=customer.get("deadline", 8.0),
                )
            )
        return tasks

    def _run_task_allocation(
        self, tasks: List[Task]
    ) -> List[Dict[str, Any]]:
        """Run sequential Contract Net rounds and award the lowest feasible bid."""
        contracts = []
        coordination_agent = next(
            agent for agent in self.mas.agents
            if agent.agent_id == "coordination_agent"
        )
        vehicle_agents = [
            agent for agent in self.mas.agents
            if agent.agent_type == AgentType.VEHICLE_AGENT
        ]

        for task in tasks:
            messages = self.communication_system._contract_net_protocol(task)
            for message in messages:
                if message.receiver_id == "broadcast":
                    # Each recipient needs its own message object and receiver ID.
                    for vehicle in vehicle_agents:
                        announcement = Message(
                            message_id=f"{message.message_id}_{vehicle.agent_id}",
                            sender_id=message.sender_id,
                            receiver_id=vehicle.agent_id,
                            message_type=message.message_type,
                            content=message.content.copy(),
                            timestamp=message.timestamp,
                            priority=message.priority,
                        )
                        self.communication_system.send_message(announcement)
                else:
                    self.communication_system.send_message(message)

            time.sleep(0.1)
            bids = [
                message.content
                for message in coordination_agent.message_queue
                if message.message_type == MessageType.BID_PROPOSAL
            ]
            coordination_agent.message_queue.clear()
            for bid in bids:
                recorded_bid = dict(bid)
                recorded_bid["task_id"] = task.task_id
                self.market_coordination.submit_bid_to_market(recorded_bid)

            if not bids:
                continue
            best_bid = min(bids, key=lambda bid: bid["bid_price"])
            contract = {
                "task_id": task.task_id,
                "agent_id": best_bid["agent_id"],
                "bid_price": best_bid["bid_price"],
                "estimated_cost": best_bid["estimated_cost"],
                "completion_time": best_bid["completion_time"],
                "contract_status": "active",
            }
            contracts.append(contract)
            self.market_coordination.marketplace["contracts"].append(
                {
                    "task_id": task.task_id,
                    "winner_id": best_bid["agent_id"],
                    "contract_price": best_bid["bid_price"],
                    "execution_time": best_bid["completion_time"],
                    "contract_status": "active",
                    "creation_time": time.time(),
                }
            )
            self.mas.task_market.append(contract)
            task.assigned_agent = best_bid["agent_id"]
            task.status = "awarded"
            assigned_vehicle = next(
                agent for agent in vehicle_agents
                if agent.agent_id == best_bid["agent_id"]
            )
            assigned_vehicle.current_load += task.requirements["capacity"]
            assert assigned_vehicle.current_load <= assigned_vehicle.capacity
        return contracts

    def _execute_contracts(
        self, contracts: List[Dict[str, Any]]
    ) -> Dict[str, Any]:
        """Group awarded tasks by vehicle and mark the simulated work complete."""
        execution_results = {
            "total_contracts": len(contracts),
            "completed_contracts": 0,
            "total_cost": 0.0,
            "total_time": 0.0,
            "agent_utilization": {},
            "contract_details": [],
        }
        agent_contracts = {}
        for contract in contracts:
            agent_contracts.setdefault(contract["agent_id"], []).append(contract)

        for agent_id, assigned_contracts in agent_contracts.items():
            agent = next(
                item for item in self.mas.agents if item.agent_id == agent_id
            )
            route = [
                int(contract["task_id"].split("_")[1])
                for contract in assigned_contracts
            ]
            agent.current_route = route
            total_cost = sum(item["bid_price"] for item in assigned_contracts)
            total_time = sum(
                item["completion_time"] for item in assigned_contracts
            )
            execution_results["total_cost"] += total_cost
            execution_results["total_time"] += total_time
            execution_results["completed_contracts"] += len(assigned_contracts)
            execution_results["agent_utilization"][agent_id] = {
                "contracts": len(assigned_contracts),
                "cost": total_cost,
                "time": total_time,
                "utilization": (
                    len(assigned_contracts) / len(contracts) if contracts else 0.0
                ),
                "route": route,
            }
            for contract in assigned_contracts:
                contract["contract_status"] = "completed"
                execution_results["contract_details"].append(
                    {
                        "task_id": contract["task_id"],
                        "agent_id": agent_id,
                        "cost": contract["bid_price"],
                        "time": contract["completion_time"],
                        "status": "completed",
                    }
                )
        return execution_results

    def _calculate_system_performance(
        self,
        contracts: List[Dict[str, Any]],
        emergent_analysis: Dict[str, Any],
    ) -> Dict[str, Any]:
        """Collect communication, market-coverage, and coordination indicators."""
        comm_stats = self.communication_system.get_communication_stats()
        market_stats = self.market_coordination.optimize_market_allocation()
        metrics = emergent_analysis["emergence_metrics"]
        return {
            "coordination_efficiency": metrics["coordination_level"],
            "communication_overhead": comm_stats["average_overhead"],
            "message_delivery_rate": comm_stats["delivery_rate"],
            "emergent_optimization": metrics["collective_intelligence"],
            "market_efficiency": market_stats["market_efficiency"],
            "system_throughput": len(contracts),
            "agent_count": len(self.mas.agents),
            "vehicle_agent_count": sum(
                agent.agent_type == AgentType.VEHICLE_AGENT
                for agent in self.mas.agents
            ),
        }

    def generate_multi_agent_report(self, results: Dict[str, Any]) -> str:
        """Format task-allocation and indicator results as a readable report."""
        execution = results["execution_results"]
        emergence = results["emergent_analysis"]
        performance = results["system_performance"]
        total = execution["total_contracts"]
        completion_rate = (
            execution["completed_contracts"] / total if total else 0.0
        )
        lines = [
            "MULTI-AGENT CVRP TASK-ALLOCATION REPORT",
            "=======================================",
            "",
            "SYSTEM OVERVIEW",
            f"- Total agents: {len(self.mas.agents)}",
            f"- Vehicle agents: {performance['vehicle_agent_count']}",
            f"- Customer agents: {sum(a.agent_type == AgentType.CUSTOMER_AGENT for a in self.mas.agents)}",
            f"- Task-allocation protocol: {self.mas.communication_protocol}",
            "- Award rule: lowest feasible Contract Net bid",
            "",
            "TASK ALLOCATION AND SIMULATED EXECUTION",
            f"- Awarded contracts: {execution['total_contracts']}",
            f"- Simulated completions: {execution['completed_contracts']}",
            f"- Completion rate: {completion_rate:.1%}",
            f"- Total bid price: {execution['total_cost']:.2f}",
            f"- Summed estimated time: {execution['total_time']:.2f} cost-model hours",
            "",
            "DESCRIPTIVE SYSTEM INDICATORS",
            f"- Coordination indicator: {performance['coordination_efficiency']:.1%}",
            f"- Communication-overhead proxy per agent: {performance['communication_overhead']:.3f}",
            f"- Message delivery rate: {performance['message_delivery_rate']:.1%}",
            f"- Combined emergent indicator: {performance['emergent_optimization']:.1%}",
            f"- Task allocation coverage: {performance['market_efficiency']:.1%}",
            "",
            "EMERGENCE INDICATORS",
            f"- Coordination level: {emergence['emergence_metrics']['coordination_level']:.1%}",
            f"- Adaptation rate from recorded histories: {emergence['emergence_metrics']['adaptation_rate']:.1%}",
            f"- Message-organization indicator: {emergence['emergence_metrics']['self_organization']:.1%}",
            f"- Combined indicator: {emergence['emergence_metrics']['collective_intelligence']:.1%}",
            "",
            "VEHICLE WORKLOAD SHARES",
        ]
        for agent_id, utilization in execution["agent_utilization"].items():
            lines.append(
                f"- {agent_id}: {utilization['contracts']} tasks, "
                f"{utilization['utilization']:.1%} of awarded tasks"
            )
        lines.extend(
            [
                "",
                "INTERPRETATION",
                "The indicators summarize this simulation's messages, task awards, and workload.",
                "They do not establish route optimality, learning, or improvement over a baseline.",
            ]
        )
        return "\n".join(lines)


print("Multi-agent CVRP system defined.")

In [ ]:
multi_agent_system = MultiAgentCVRPSystem()

# The supplied customer data define the example instance.
problem_instance = {
    "customers": [
        {"id": 1, "location": (5, 10), "demand": 10, "priority": 1, "deadline": 4.0},
        {"id": 2, "location": (15, 5), "demand": 8, "priority": 2, "deadline": 6.0},
        {"id": 3, "location": (10, 15), "demand": 12, "priority": 1, "deadline": 5.0},
        {"id": 4, "location": (20, 10), "demand": 6, "priority": 3, "deadline": 7.0},
        {"id": 5, "location": (8, 12), "demand": 9, "priority": 2, "deadline": 6.0},
        {"id": 6, "location": (12, 8), "demand": 11, "priority": 1, "deadline": 5.0},
        {"id": 7, "location": (18, 12), "demand": 7, "priority": 2, "deadline": 6.0},
        {"id": 8, "location": (6, 8), "demand": 10, "priority": 1, "deadline": 4.0},
        {"id": 9, "location": (14, 18), "demand": 8, "priority": 3, "deadline": 8.0},
        {"id": 10, "location": (16, 6), "demand": 9, "priority": 2, "deadline": 6.0},
        {"id": 11, "location": (4, 14), "demand": 11, "priority": 1, "deadline": 5.0},
        {"id": 12, "location": (22, 8), "demand": 6, "priority": 3, "deadline": 7.0},
    ]
}

results = multi_agent_system.solve_cvrp_multi_agent(problem_instance)
multi_agent_report = multi_agent_system.generate_multi_agent_report(results)
print(multi_agent_report)

execution_results = results["execution_results"]
system_performance = results["system_performance"]
emergent_analysis = results["emergent_analysis"]

# These checks connect the reported summaries to the awarded task records.
assert execution_results["completed_contracts"] == len(results["contracts"])
assert execution_results["total_contracts"] == len(results["contracts"])
assert all(
    agent.current_load <= agent.capacity
    for agent in multi_agent_system.mas.agents
    if agent.agent_type == AgentType.VEHICLE_AGENT
)
assert sum(
    detail["cost"] for detail in execution_results["contract_details"]
) == execution_results["total_cost"]

print("\nSYSTEM PERFORMANCE SUMMARY")
print("=" * 70)
print(f"Vehicle agents: {system_performance['vehicle_agent_count']}")
print(f"Coordination indicator: {system_performance['coordination_efficiency']:.1%}")
print(f"Communication-overhead proxy per agent: {system_performance['communication_overhead']:.3f}")
print(f"Message delivery rate: {system_performance['message_delivery_rate']:.1%}")
print(f"Task allocation coverage: {system_performance['market_efficiency']:.1%}")
print(f"Combined emergent indicator: {system_performance['emergent_optimization']:.1%}")

print("\nEMERGENCE INDICATORS")
for name, value in emergent_analysis["emergence_metrics"].items():
    print(f"{name.replace('_', ' ').title()}: {value:.1%}")
print("Identified patterns:", multi_agent_system.emergent_behavior.identify_emergent_patterns())

In [ ]:
def visualize_multi_agent_system(multi_agent_system, results):
    """Plot agent roles, indicators, workload, and model-scope notes."""
    execution = results["execution_results"]
    emergence = results["emergent_analysis"]
    performance = results["system_performance"]
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(
        2, 2, figsize=(15, 12)
    )

    # Panel 1: agent-role composition.
    agent_types = {}
    for agent in multi_agent_system.mas.agents:
        label = agent.agent_type.value.replace("_", " ").title()
        agent_types[label] = agent_types.get(label, 0) + 1
    ax1.set_title("Agent Roles in the Simulation")
    ax1.pie(
        list(agent_types.values()),
        labels=list(agent_types.keys()),
        autopct="%1.1f%%",
        startangle=90,
    )
    ax1.axis("equal")

    # Panel 2: indicators are shown on a percentage scale for readability.
    metric_names = [
        "Coordination\nindicator",
        "Combined\nemergent indicator",
        "Task allocation\ncoverage",
        "Message delivery\nrate",
    ]
    metric_values = [
        performance["coordination_efficiency"] * 100,
        performance["emergent_optimization"] * 100,
        performance["market_efficiency"] * 100,
        performance["message_delivery_rate"] * 100,
    ]
    ax2.set_title("Descriptive System Indicators")
    bars = ax2.bar(metric_names, metric_values, color="steelblue", alpha=0.8)
    ax2.set_ylabel("Indicator value (%)")
    ax2.set_ylim(0, 100)
    ax2.grid(axis="y", alpha=0.3)
    for bar, value in zip(bars, metric_values):
        ax2.text(
            bar.get_x() + bar.get_width() / 2,
            min(bar.get_height() + 2, 98),
            f"{value:.1f}%",
            ha="center",
            va="bottom",
            fontsize=9,
        )

    # Panel 3: each point shows one vehicle's share of awarded tasks.
    ax3.set_title("Vehicle Workload Distribution")
    utilization = execution["agent_utilization"]
    if utilization:
        names = list(utilization)
        shares = [utilization[name]["utilization"] * 100 for name in names]
        counts = [utilization[name]["contracts"] for name in names]
        points = ax3.scatter(
            shares,
            counts,
            s=90,
            c=range(len(names)),
            cmap="viridis",
        )
        colorbar = fig.colorbar(points, ax=ax3)
        colorbar.set_label("Vehicle index in plotted data")
        for name, share, count in zip(names, shares, counts):
            ax3.annotate(name.replace("vehicle_agent_", ""), (share, count))
    else:
        ax3.text(0.5, 0.5, "No tasks were awarded.", ha="center", va="center")
    ax3.set_xlabel("Share of awarded tasks (%)")
    ax3.set_ylabel("Number of assigned tasks")
    ax3.grid(True, alpha=0.3)

    # Panel 4: state the scope alongside the metrics to avoid overinterpretation.
    ax4.set_title("Interpretation and Scope")
    ax4.axis("off")
    metrics = emergence["emergence_metrics"]
    dashboard_text = (
        "Descriptive indicators\n"
        "──────────────────────────────\n"
        f"Coordination: {metrics['coordination_level']:.1%}\n"
        f"Adaptation from histories: {metrics['adaptation_rate']:.1%}\n"
        f"Message organization: {metrics['self_organization']:.1%}\n"
        f"Combined indicator: {metrics['collective_intelligence']:.1%}\n\n"
        f"Awarded tasks: {execution['total_contracts']}\n"
        f"Simulated completions: {execution['completed_contracts']}\n"
        f"Communication overhead proxy: {performance['communication_overhead']:.3f}\n\n"
        "Scope limits\n"
        "• Bid costs are simplified proxies.\n"
        "• Customer coordinates do not determine bids.\n"
        "• Route distance and time windows are not optimized.\n"
        "• Completion is simulated, not physically observed."
    )
    ax4.text(
        0.04,
        0.96,
        dashboard_text,
        fontsize=10,
        verticalalignment="top",
        fontfamily="monospace",
        bbox={"boxstyle": "round", "facecolor": "honeydew", "alpha": 0.8},
    )

    fig.suptitle("Multi-Agent CVRP Task-Allocation Simulation", fontsize=15)
    fig.tight_layout()
    plt.show()


visualize_multi_agent_system(multi_agent_system, results)

## Why use a multi-agent perspective?

A multi-agent system represents a logistics process as interacting decision-making entities. Vehicle agents can report their own capacity and costs, customer agents can represent requests, and a coordination agent can organize task announcements and awards. This representation is useful when the system being studied is distributed or when local decisions and communication are themselves part of the problem.

In this example, the Contract Net Protocol supplies the allocation sequence: announce a task, receive bids, select the lowest feasible bid, and update the assigned vehicle's load. The market subsystem stores bids and awards and reports task-allocation coverage; it does not determine the winner through an independent auction. The indicators summarize the resulting messages, workload, and recorded performance histories.

## Potential strengths and limitations

**Potential strengths**
- **Modularity:** Vehicle, customer, coordination, and monitoring responsibilities are represented separately.
- **Local feasibility checks:** A vehicle can decline a task when its remaining capacity is insufficient.
- **Explicit communication:** Announcements and bids make the coordination sequence inspectable.
- **Extensibility:** Additional constraints or agent roles can be represented within the same general agent-based architecture.

**Limitations and implementation costs**
- **Communication and coordination:** Distributed decisions require messages, protocol rules, and consistent state updates.
- **Global performance is not guaranteed:** Selecting the lowest bid task by task does not prove that the overall set of routes is optimal.
- **Limited realism:** This simulation does not model a road network, route sequencing costs, time-window feasibility, or uncertain execution.
- **Interpretation of indicators:** The reported coordination and emergence metrics are constructed summaries, not validated measures of operational performance.
- **Operational complexity:** More agent roles and interactions can make testing, monitoring, and debugging more demanding.

## When this perspective may be useful

Agent-based coordination is worth studying when decisions are distributed across vehicles, facilities, or organizations; when local information and negotiation matter; or when communication and adaptation are central research questions. For practical routing decisions, the scope of the cost model and the feasibility of complete routes must be examined before interpreting an allocation as a CVRP solution.